# Book Recommendation Dataset — Mục 1–5
Theo phần III “Suggestions on how to do the exercise” trong Mid-Term.pdf.
Notebook thực hiện kiểm tra, ghép, làm sạch dữ liệu, EDA và xây dựng mô hình phân loại.
Chạy từ đầu đến cuối bằng Run All. Đặt thư mục DATA_SET cạnh notebook; trên Colab, tải DATA_SET lên /content/DATA_SET hoặc sửa DATA_DIR.
Thư viện cần có: pandas, numpy, matplotlib, scikit-learn và IPython (có sẵn trên Colab).
Dữ liệu gốc được giữ nguyên. Các ngưỡng làm sạch và nhãn liked là quy ước của bài làm.


In [130]:
# Nhập thư viện: Path xử lý đường dẫn; pandas xử lý bảng; numpy hỗ trợ tính toán; display hiển thị bảng.
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
# Chọn thư mục chứa dữ liệu CSV; kiểm tra thư mục tồn tại trước khi đọc.
DATA_DIR = Path("DATA_SET")
assert DATA_DIR.is_dir(), "Cần đặt thư mục DATA_SET trong thư mục làm việc hoặc sửa DATA_DIR."


## 1. Data Inspection
Đọc tự động ba bảng CSV. Kiểm tra UTF-8 bằng chế độ strict trên toàn bộ tệp; giữ ISBN dạng chuỗi để bảo toàn số 0 đầu và ký tự X. Ba bảng khác cấu trúc nên dùng merge thay vì concat.

In [131]:
# Tạo từ điển lưu ba bảng dữ liệu theo tên Books, Users và Ratings.
tables = {}
# Lần lượt đọc từng tệp CSV trong thư mục dữ liệu.
for name in ["Books", "Users", "Ratings"]:
    path = DATA_DIR / f"{name}.csv"
    # Đọc toàn bộ tệp với UTF-8 strict để phát hiện lỗi mã hóa; pass chỉ bỏ qua nội dung từng dòng.
    with path.open(encoding="utf-8", errors="strict") as stream:
        for _ in stream:
            pass
    # Nạp CSV vào DataFrame; giữ ISBN dạng chuỗi để không mất số 0 đầu hoặc ký tự X.
    tables[name] = pd.read_csv(path, encoding="utf-8", dtype={"ISBN": "string"}, low_memory=False)
# Lấy ba DataFrame từ từ điển để dùng ở các bước sau.
books, users, ratings = [tables[n] for n in ["Books", "Users", "Ratings"]]
# Tóm tắt mỗi bảng: shape[0] là số dòng, shape[1] là số cột; duplicated đánh dấu dòng trùng.
summary = pd.DataFrame([{"Bảng": n, "Số dòng": d.shape[0], "Số cột": d.shape[1], "Dòng trùng": int(d.duplicated().sum())} for n,d in tables.items()])
display(summary)
# Hiển thị kiểu dữ liệu và số ô thiếu của từng cột trong mỗi bảng.
for name, frame in tables.items():
    print("\n", name)
    # dtypes lấy kiểu từng cột; astype(str) chuyển thông tin kiểu thành chuỗi, không đổi dữ liệu gốc.
    display(pd.DataFrame({"Kiểu": frame.dtypes.astype(str), "Thiếu": frame.isna().sum()}))


,Bảng,Số dòng,Số cột,Dòng trùng
0,Books,271360,8,0
1,Users,278858,3,0
2,Ratings,1149780,3,0



 Books


,Kiểu,Thiếu
ISBN,string,0
Book-Title,str,0
Book-Author,str,2
Year-Of-Publication,str,0
Publisher,str,2
Image-URL-S,str,0
Image-URL-M,str,0
Image-URL-L,str,3



 Users


,Kiểu,Thiếu
User-ID,int64,0
Location,str,0
Age,float64,110762



 Ratings


,Kiểu,Thiếu
User-ID,int64,0
ISBN,string,0
Book-Rating,int64,0


In [132]:
# Kiểm tra khóa ISBN và User-ID không bị thiếu và không trùng; assert báo lỗi nếu điều kiện sai.
assert books["ISBN"].notna().all() and books["ISBN"].is_unique
assert users["User-ID"].notna().all() and users["User-ID"].is_unique
# Đếm các dòng đánh giá trùng cặp người dùng–sách.
print("Trùng cặp User-ID, ISBN:", ratings.duplicated(["User-ID", "ISBN"]).sum())
# isin kiểm tra ISBN có trong bảng sách; ~ đảo kết quả; sum đếm các đánh giá không tìm thấy sách.
print("Đánh giá thiếu sách:", (~ratings["ISBN"].isin(books["ISBN"])).sum())
# Đếm các đánh giá có User-ID không xuất hiện trong bảng người dùng.
print("Đánh giá thiếu người dùng:", (~ratings["User-ID"].isin(users["User-ID"])).sum())
# Đếm số lượt cho từng mức điểm, sắp theo điểm rồi hiển thị dưới dạng bảng.
display(ratings.groupby("Book-Rating").size().rename("Số lượt").to_frame())
# Chuyển năm xuất bản sang số; errors="coerce" biến giá trị không chuyển được thành NaN.
year = pd.to_numeric(books["Year-Of-Publication"], errors="coerce")
# loc chọn những dòng có năm thiếu hoặc không chuyển được thành số và các cột cần xem xét.
display(books.loc[year.isna(), ["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]])
# Kiểm tra tuổi nhỏ nhất và lớn nhất để phát hiện giá trị bất thường, ví dụ 0 hoặc 244.
print("Tuổi nhỏ nhất/lớn nhất:", users["Age"].min(), users["Age"].max())


Trùng cặp User-ID, ISBN: 0
Đánh giá thiếu sách: 118644
Đánh giá thiếu người dùng: 0


,Số lượt
Book-Rating,
0,716109
1,1770
2,2759
3,5996
4,8904
5,50974
6,36924
7,76457
8,103736


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher
209538,078946697X,"DK Readers: Creating the X-Men, How It All Beg...",2000,DK Publishing Inc,http://images.amazon.com/images/P/078946697X.0...
220731,2070426769,"Peuple du ciel, suivi de 'Les Bergers\"";Jean-M...",2003,Gallimard,http://images.amazon.com/images/P/2070426769.0...
221678,0789466953,"DK Readers: Creating the X-Men, How Comic Book...",2000,DK Publishing Inc,http://images.amazon.com/images/P/0789466953.0...


Tuổi nhỏ nhất/lớn nhất: 0.0 244.0


## 2. Data Loading and Integration
Ratings là bảng trung tâm, mỗi dòng là một lượt đánh giá. Ratings có quan hệ nhiều–một với Users qua cột User-ID và với Books qua cột ISBN: một người dùng có thể có nhiều lượt đánh giá, một sách có thể được đánh giá nhiều lần. Dùng left join để giữ toàn bộ lượt đánh giá, kể cả khi không tìm thấy người dùng hoặc sách tương ứng. Tham số `validate='many_to_one'` kiểm tra khóa ở bảng Users hoặc Books không bị trùng; nếu bị trùng thì báo lỗi để tránh ghép làm tăng số dòng. Chỉ ghép các cột cần phân tích để giảm bộ nhớ.

In [139]:
# Chọn các cột thông tin sách cần ghép để giảm bộ nhớ.
book_columns = ["ISBN", "Book-Title", "Book-Author", "Year-Of-Publication", "Publisher"]
# Ghép đánh giá với sách theo ISBN; left giữ mọi đánh giá, many_to_one yêu cầu ISBN bên sách không trùng.
integrated_raw = ratings.merge(books[book_columns], on="ISBN", how="left", validate="many_to_one", indicator="book_match")
# Ghép tiếp người dùng theo User-ID; các cột indicator ghi trạng thái khớp của mỗi lần ghép.
integrated_raw = integrated_raw.merge(users, on="User-ID", how="left", validate="many_to_one", indicator="user_match")
# Đảm bảo việc ghép không làm thay đổi số dòng đánh giá.
assert len(integrated_raw) == len(ratings)
# Xem kích thước bảng ghép và số dòng khớp/không khớp với từng bảng phụ.
print("Kích thước bảng ghép:", integrated_raw.shape)
display(integrated_raw["book_match"].value_counts())
display(integrated_raw["user_match"].value_counts())
display(integrated_raw.head())


Kích thước bảng ghép: (1149780, 11)


book_match
both          1031136
left_only      118644
right_only          0
Name: count, dtype: int64

user_match
both          1149780
left_only           0
right_only          0
Name: count, dtype: int64

,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,book_match,Location,Age,user_match
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,both,"tyler, texas, usa",NaN,both
1,276726,0155061224,5,Rites of Passage,Judith Rae,2001,Heinle,both,"seattle, washington, usa",NaN,both
2,276727,0446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,both,"h, new south wales, australia",16.0,both
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,both,"rijeka, n/a, croatia",16.0,both
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,both,"rijeka, n/a, croatia",16.0,both


## 3. Data Cleaning
- Bỏ dòng trùng hoàn toàn; nếu còn nhiều điểm khác nhau cho cùng cặp người dùng–sách thì dừng để xem xét, không tự chọn một điểm.
- Loại riêng 3 dòng sách có năm không phải số khỏi bảng phân tích vì có dấu hiệu lệch trường; không suy đoán để sửa. Các đánh giá liên quan vẫn ở bảng ghép, được đánh dấu không có metadata sách sạch.
- Quy ước tuổi hợp lệ 5–100 và năm xuất bản 1450–2026; đây là ngưỡng đề xuất, không phải kết luận mọi giá trị ngoài khoảng đều sai. Năm 0 coi là chưa biết. Giữ giá trị thiếu dạng số, tạo cờ thiếu; chưa điền median trước khi chia train/test để tránh rò rỉ dữ liệu.
- Điền Unknown cho tác giả, nhà xuất bản và vị trí thiếu/rỗng; chuẩn hóa khoảng trắng. ISBN luôn là chuỗi.
- Giữ điểm 0 ở bảng tổng hợp. Với tập phân loại đề xuất, chỉ lấy điểm 1–10 và đặt liked = 1 nếu điểm ≥ 8. Ý nghĩa của điểm 0 cần xác nhận theo mô tả dataset; không tự gán 0 thành nhãn không thích.


In [134]:
# Bỏ các dòng trùng hoàn toàn và tạo bản sao để giữ nguyên ba bảng gốc.
books_clean = books.drop_duplicates().copy()
users_clean = users.drop_duplicates().copy()
ratings_clean = ratings.drop_duplicates().copy()
# Kiểm tra khóa sách duy nhất và không còn cặp người dùng–sách trùng cần xem xét.
assert books_clean["ISBN"].is_unique
assert not ratings_clean.duplicated(["User-ID", "ISBN"]).any(), "Cần xem xét đánh giá xung đột."
# Chuyển năm sang số; giá trị thiếu hoặc không chuyển được sẽ thành NaN.
parsed_year = pd.to_numeric(books_clean["Year-Of-Publication"], errors="coerce")
# Tách riêng các dòng có năm không chuyển được hoặc bị thiếu; loc lọc dòng theo điều kiện.
books_quarantine = books_clean.loc[parsed_year.isna()].copy()
books_clean = books_clean.loc[parsed_year.notna()].copy()
# Chuyển lại cột năm của bảng sách còn giữ để chuẩn hóa kiểu số.
y = pd.to_numeric(books_clean["Year-Of-Publication"], errors="coerce")
# Chỉ giữ năm 1450–2026; where thay các năm ngoài khoảng (kể cả 0) bằng giá trị thiếu.
books_clean["Year-Of-Publication"] = y.where(y.between(1450, 2026)).astype("Int64")
# Tạo cột cờ: True nếu năm bị thiếu sau xử lý. Int64 phía trên cho phép lưu số nguyên có giá trị thiếu.
books_clean["Year_missing"] = books_clean["Year-Of-Publication"].isna()
# Chuyển tuổi sang số; giá trị không chuyển được thành giá trị thiếu.
a = pd.to_numeric(users_clean["Age"], errors="coerce")
# Theo quy ước của bài, chỉ giữ tuổi 5–100; tuổi như 244 thành <NA>, không xóa dòng người dùng.
users_clean["Age"] = a.where(a.between(5, 100)).astype("Float64")
# Đánh dấu người dùng có tuổi thiếu sau xử lý.
users_clean["Age_missing"] = users_clean["Age"].isna()
# Làm sạch các cột chữ: bỏ khoảng trắng ở hai đầu, đổi chuỗi rỗng thành thiếu rồi điền Unknown.
for frame, cols in [(books_clean, ["Book-Title", "Book-Author", "Publisher"]), (users_clean, ["Location"])]:
    for col in cols:
        frame[col] = frame[col].astype("string").str.strip().replace("", pd.NA).fillna("Unknown")
# Chuẩn hóa User-ID thành số nguyên ở cả hai bảng; errors="raise" báo lỗi nếu không chuyển được.
for frame in [users_clean, ratings_clean]:
    frame["User-ID"] = pd.to_numeric(frame["User-ID"], errors="raise").astype("int64")
# Chuyển điểm đánh giá sang số; kiểm tra mọi điểm nằm trong khoảng 0–10 và là số nguyên.
ratings_clean["Book-Rating"] = pd.to_numeric(ratings_clean["Book-Rating"], errors="raise")
assert ratings_clean["Book-Rating"].between(0, 10).all()
assert (ratings_clean["Book-Rating"] % 1 == 0).all()
# Lưu điểm bằng int8 vì khoảng 0–10 nhỏ, giúp tiết kiệm bộ nhớ.
ratings_clean["Book-Rating"] = ratings_clean["Book-Rating"].astype("int8")
# Báo số sách tách riêng, số tuổi thiếu và số năm thiếu sau khi làm sạch.
print("Sách tách riêng:", len(books_quarantine))
print("Tuổi thiếu sau xử lý:", users_clean["Age"].isna().sum())
print("Năm thiếu sau xử lý:", books_clean["Year-Of-Publication"].isna().sum())


Sách tách riêng: 3
Tuổi thiếu sau xử lý: 112010
Năm thiếu sau xử lý: 4631


In [135]:
# Ghép lại các bảng đã làm sạch; left vẫn giữ các đánh giá chưa tìm thấy thông tin sách sạch.
data_clean = ratings_clean.merge(books_clean[book_columns + ["Year_missing"]], on="ISBN", how="left", validate="many_to_one", indicator="book_match")
data_clean = data_clean.merge(users_clean, on="User-ID", how="left", validate="many_to_one", indicator="user_match")
# Kiểm tra bảng ghép giữ nguyên số đánh giá và không sinh cặp người dùng–sách trùng.
assert len(data_clean) == len(ratings_clean)
assert not data_clean.duplicated(["User-ID", "ISBN"]).any()
# Tạo tập phân loại chỉ từ điểm 1–10; điểm 0 vẫn được giữ trong data_clean.
classification_data = data_clean.loc[data_clean["Book-Rating"].between(1, 10)].copy()
# Tạo nhãn liked: điểm từ 8 trở lên là 1 (thích), điểm 1–7 là 0.
classification_data["liked"] = (classification_data["Book-Rating"] >= 8).astype("int8")
# Xem kích thước dữ liệu và số đánh giá không có thông tin sách sạch (left_only).
print("Bảng sạch:", data_clean.shape)
print("Lượt đánh giá chưa có metadata sách sạch:", data_clean["book_match"].eq("left_only").sum())
print("Tập phân loại đề xuất:", classification_data.shape)
# Đếm số mẫu ở mỗi nhãn 0 và 1 để xem phân bố nhãn.
print(classification_data["liked"].value_counts().sort_index().to_string())
# Hiển thị 5 dòng đầu của bảng sạch để kiểm tra kết quả.
display(data_clean.head())


Bảng sạch: (1149780, 13)
Lượt đánh giá chưa có metadata sách sạch: 118648
Tập phân loại đề xuất: (433671, 14)
liked
0    183784
1    249887


,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Year_missing,book_match,Location,Age,Age_missing,user_match
0,276725,034545104X,0,Flesh Tones: A Novel,M. J. Rose,2002,Ballantine Books,False,both,"tyler, texas, usa",<NA>,True,both
1,276726,0155061224,5,Rites of Passage,Judith Rae,2001,Heinle,False,both,"seattle, washington, usa",<NA>,True,both
2,276727,0446520802,0,The Notebook,Nicholas Sparks,1996,Warner Books,False,both,"h, new south wales, australia",16.0,False,both
3,276729,052165615X,3,Help!: Level 1,Philip Prowse,1999,Cambridge University Press,False,both,"rijeka, n/a, croatia",16.0,False,both
4,276729,0521795028,6,The Amsterdam Connection : Level 4 (Cambridge ...,Sue Leather,2001,Cambridge University Press,False,both,"rijeka, n/a, croatia",16.0,False,both


### Chuyển sang EDA và xây dựng mô hình
Các bảng sạch được dùng ở phần 4–5 dưới đây. Đây là bài toán dự đoán một lượt đánh giá có điểm ≥ 8 hay không, trong nhóm đánh giá có điểm 1–10. Không suy rộng kết quả thành hành vi của tất cả người dùng hoặc khả năng họ sẽ mua/đọc sách. Điểm 0 được giữ ở bảng tổng hợp nhưng chưa dùng làm nhãn phân loại vì ý nghĩa cần được xác nhận.